In [29]:
%pip install pandas 
%pip install numpy 

You should consider upgrading via the '/Users/niranjanhebli/Desktop/Week3/week3/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/niranjanhebli/Desktop/Week3/week3/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
import numpy as np
import json
import random

### Create Messy Survey Dataset

In [31]:
def create_dataset():

    first_names = ["Alice","Bob","Charlie","David","Eva","Frank","Grace","Hannah","Ian","Jack",
                   "Karen","Leo","Mia","Nina","Owen","Paul","Quinn","Rita","Sam","Tina",
                   "Uma","Victor","Wendy","Xavier","Yara","Zane"]

    cities = ["New York","Los Angeles","Chicago","Houston","Phoenix","Seattle",
              "Boston","Miami","Denver","Austin"]

    genders = ["Male","Female","Other"]

    recommendations = ["Yes","No"]

    comments = ["Good service","Bad support","Excellent","Average","Not satisfied",
                "Very happy","Could improve","Great experience"]

    rows = []

    for i in range(60):

        name = random.choice(first_names)

        # introduce whitespace and casing problems
        if random.random() < 0.3:
            name = " " + name + " "
        if random.random() < 0.3:
            name = name.upper()

        age = random.choice([random.randint(18,70), -5, 200, "unknown", None])

        gender = random.choice(genders)
        if random.random() < 0.3:
            gender = gender.lower()
        if random.random() < 0.2:
            gender = gender + " "

        city = random.choice(cities)
        if random.random() < 0.3:
            city = city.lower()
        if random.random() < 0.2:
            city = " " + city

        score = random.choice([1,2,3,4,5,None,"NaN"])

        recommend = random.choice(recommendations)
        if random.random() < 0.3:
            recommend = recommend.lower()
        if random.random() < 0.2:
            recommend = recommend + " "

        comment = random.choice(comments)

        if random.random() < 0.2:
            comment = ""
        if random.random() < 0.1:
            comment = None

        rows.append({
            "respondent_id": i+1,
            "name": name,
            "age": age,
            "gender": gender,
            "city": city,
            "satisfaction_score": score,
            "recommend": recommend,
            "comments": comment
        })

    df = pd.DataFrame(rows)

    # introduce duplicates intentionally
    df = pd.concat([df, df.sample(5)], ignore_index=True)

    return df

In [32]:
df = create_dataset()

In [33]:
df.to_csv("survey_responses.csv", index=False)

### Detect Data Quality Issues

In [34]:
def detect_issues(df):

    report = {}

    report["total_rows"] = len(df)
    report["total_missing"] = int(df.isna().sum().sum())
    report["missing_per_column"] = df.isna().sum().to_dict()
    report["duplicate_count"] = int(df.duplicated().sum())

    wrong_types = {
        "age_non_numeric": int(pd.to_numeric(df["age"], errors="coerce").isna().sum()),
        "score_non_numeric": int(pd.to_numeric(df["satisfaction_score"], errors="coerce").isna().sum())
    }

    report["wrong_types"] = wrong_types

    invalid_age = df[
        (pd.to_numeric(df["age"], errors="coerce") < 0) |
        (pd.to_numeric(df["age"], errors="coerce") > 120)
    ].shape[0]

    report["invalid_values"] = {
        "invalid_age": int(invalid_age)
    }

    return report


In [35]:
report = detect_issues(df)

### Clean Data

In [36]:
def clean_data(df):

    # Replace hidden NaN markers
    df.replace(["", "NaN", "None", "unknown"], np.nan, inplace=True)

    # Standardize text columns
    text_cols = ["name","gender","city","recommend","comments"]

    for col in text_cols:
        df[col] = df[col].astype(str).str.strip().str.lower()

    # Convert numeric columns
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["satisfaction_score"] = pd.to_numeric(df["satisfaction_score"], errors="coerce")

    # Remove invalid age
    df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan

    # Fill missing values
    df["age"].fillna(df["age"].median(), inplace=True)  
    # median used because age distribution may contain outliers

    df["gender"].fillna("unknown", inplace=True)  
    # categorical column filled with explicit label

    df["city"].fillna("unknown", inplace=True)  
    # keeps row but marks missing location

    df["satisfaction_score"].fillna(df["satisfaction_score"].mean(), inplace=True)  
    # mean keeps overall rating distribution

    df["recommend"].fillna("no", inplace=True)  
    # survey assumption: missing recommendation treated as negative

    df["comments"].fillna("no_comment", inplace=True)  
    # ensures text column consistency

    # Drop rows missing critical identifier
    df.dropna(subset=["respondent_id"], inplace=True)

    # Remove duplicates
    df.drop_duplicates(inplace=True)

    return df

In [37]:
df_clean = clean_data(df.copy())

/var/folders/v5/3ngzhn6141146q6rw7_b54cm0000gn/T/ipykernel_35434/1280850201.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(["", "NaN", "None", "unknown"], np.nan, inplace=True)
/var/folders/v5/3ngzhn6141146q6rw7_b54cm0000gn/T/ipykernel_35434/1280850201.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the

### Compare Before vs After

In [38]:
def compare_stats(df_before, df_after):

    stats = pd.DataFrame({
        "Metric": ["Rows","Null Values","Memory Usage"],
        "Before": [
            df_before.shape[0],
            df_before.isna().sum().sum(),
            df_before.memory_usage(deep=True).sum()
        ],
        "After": [
            df_after.shape[0],
            df_after.isna().sum().sum(),
            df_after.memory_usage(deep=True).sum()
        ]
    })

    print("\nBefore vs After Cleaning\n")
    print(stats)

    print("\nDtypes Before\n", df_before.dtypes)
    print("\nDtypes After\n", df_after.dtypes)


In [39]:
compare_stats(df, df_clean)


Before vs After Cleaning

         Metric  Before  After
0          Rows      65     60
1   Null Values      31      0
2  Memory Usage   25797  20711

Dtypes Before
 respondent_id          int64
name                  object
age                   object
gender                object
city                  object
satisfaction_score    object
recommend             object
comments              object
dtype: object

Dtypes After
 respondent_id           int64
name                   object
age                   float64
gender                 object
city                   object
satisfaction_score    float64
recommend              object
comments               object
dtype: object


In [40]:
df_clean.to_csv("cleaned_survey.csv", index=False)

with open("data_quality_report.json","w") as f:
    json.dump(report, f, indent=4)

print("\nExported:")
print("cleaned_survey.csv")
print("data_quality_report.json")


Exported:
cleaned_survey.csv
data_quality_report.json
